# Stage 2 Notebook 18 - Exp2M CLRKD + LineIoU regression + sqrt target rescaling

Independent ablation alongside Exp2L (NB17). Both descend from Exp2K's IoU-regression cls formulation but address its 'all targets ~ 0' collapse via different fixes.

Exp2M = Exp2K + **sqrt rescaling on the IoU target**. Target becomes `sqrt(LineIoU)` instead of `LineIoU`:

- iou=0.04 -> target=0.20 (was 0.04)
- iou=0.16 -> target=0.40 (was 0.16)
- iou=0.36 -> target=0.60 (was 0.36)
- iou=0.81 -> target=0.90 (was 0.81)

Effect: priors with small but non-zero IoU now have meaningful target signals (0.20+ instead of 0.04). The 'predict 0 for everyone' attractor of Exp2K, where ~95% of priors had target ~ 0 and dragged all logits down, is broken because more priors carry non-trivial supervision.

Single config knob change vs Exp2K: `lineiou_target_pow: 0.5`. cls_loss_type stays `bce`. Everything else identical.

Compared to Exp2L (QFL): Exp2L modifies the loss *function* to weight by error magnitude; Exp2M modifies the target *distribution* to spread it away from zero. Both could plausibly fix the same underlying problem; the empirical winner is what we run them to find out. They are ABLATIONS of the same Exp2K parent and are independent of each other -- run either or both.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through BCE on sqrt-rescaled IoU target.
# Must print 'OK exp13_*.yaml' with shapes lane_shape=(1, 16, 72, 2)
# det_shape=(1, 4, 4) before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_smoke.log
OK exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=9.8262 det_loss=3.5474 grad_cos=-0.2312 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4992882311344147, 'gate/lane_mean': 0.5011213421821594, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml --curve-tar /content/drive/M

0

## What to watch in Exp2M training

Pass criteria are the same as Exp2L. Specifically at epoch 10:

- `pred_lanes / batch` > 100 (Exp2K was 0).
- `val/lane_exist_pos_score_mean - val/lane_exist_neg_score_mean` >= 0.10. The two distributions should separate.
- `val/lane_exist_best_f1` >= 0.30.
- Geometry holds: `point_mae <= 0.34`, `matched_line_iou >= 0.40`.
- `val/lane/clrkd_style_f1` rises noticeably above the ~0.02 floor.

Failure signal: pred_lanes still 0 -> sqrt rescaling not aggressive enough. Try `lineiou_target_pow: 0.25` (4-th root) or fall back to QFL.

After short10, run NB08 to compare Exp2K / Exp2L / Exp2M side-by-side. The winner becomes the parent for Exp2N (lane decode + NMS + proper lane-F1 metric).